# Data sources: everything `add_*` accepts

Every layer method — `add_markers`, `add_circle_markers`, `add_line`,
`add_polygon`, `add_collection` — takes the same range of inputs. This notebook
shows each one with the smallest dataset that proves it. Sections are independent:
each needs only the library it demonstrates (swiftmap itself depends only on
`anywidget` and `numpy`), so skip any section for a library you don't use.

A method that cannot read what you pass **warns and adds nothing** rather than
raising — the last section shows that on purpose.

In [ ]:
import numpy as np
import pandas as pd
from swiftmap import Map

## 1. Pandas: lat/lon columns found by name

Candidate names are `lat`/`latitude`/`y` and `lon`/`longitude`/`x`/`lng`. Every
other column becomes a feature property (popups, tooltips, `color_col`, ...).

In [ ]:
towns = pd.DataFrame({
    "lat": [36.14, 36.01, 35.89],
    "lon": [-5.45, -5.60, -5.32],
    "town": ["San Roque", "Algeciras", "Ceuta"],
    "pop_k": [33, 122, 83],
})
m = Map()
m.add_circle_markers(towns, name="Towns")
m

When your columns are named something the guess would never find, say so:

In [ ]:
odd = towns.rename(columns={"lat": "phi", "lon": "lam"})
m = Map()
m.add_circle_markers(odd, lat_col="phi", lon_col="lam", name="Odd names")
m

## 2. Polars: the same call

In [ ]:
import polars as pl

m = Map()
m.add_circle_markers(pl.from_pandas(towns), name="Towns (polars)")
m

## 3. Long format: one row per vertex

`line_id_col` groups rows into separate lines and `order_col` sequences the
vertices — the natural shape of track logs. `name` matching a column names each
line from its own value.

In [ ]:
rng = np.random.default_rng(3)
steps = 40
tracks = pd.DataFrame({
    "track_id": np.repeat(["Vessel A", "Vessel B"], steps),
    "step": np.tile(np.arange(steps), 2),
    "lat": np.concatenate([
        36.00 + np.cumsum(rng.normal(0.002, 0.004, steps)),
        35.95 + np.cumsum(rng.normal(0.003, 0.004, steps)),
    ]),
    "lon": np.concatenate([
        -5.80 + np.cumsum(rng.normal(0.010, 0.006, steps)),
        -5.75 + np.cumsum(rng.normal(0.008, 0.006, steps)),
    ]),
})
m = Map()
m.add_line(tracks, line_id_col="track_id", order_col="step",
           name="track_id", weight=3)
m

Polygons work the same way with `shape_id_col`:

In [ ]:
zones = pd.DataFrame({
    "zone_id": ["North", "North", "North", "North",
                "South", "South", "South", "South"],
    "vertex": [0, 1, 2, 3] * 2,
    "lat": [36.10, 36.10, 36.16, 36.16, 35.98, 35.98, 36.04, 36.04],
    "lon": [-5.55, -5.40, -5.40, -5.55, -5.55, -5.40, -5.40, -5.55],
})
m = Map()
m.add_polygon(zones, shape_id_col="zone_id", order_col="vertex",
              name="zone_id", fill_opacity=0.3)
m

## 4. Wide vertex columns

`lat1, lon1, lat2, lon2, ...` — one shape per row, vertices read left to right.

In [ ]:
wide = pd.DataFrame({
    "name": ["Box 1", "Box 2"],
    "lat1": [36.05, 36.20], "lon1": [-5.90, -5.95],
    "lat2": [36.05, 36.20], "lon2": [-5.80, -5.85],
    "lat3": [36.12, 36.27], "lon3": [-5.80, -5.85],
    "lat4": [36.12, 36.27], "lon4": [-5.90, -5.95],
})
m = Map()
m.add_polygon(wide, name="name")
m

## 5. WKT geometry columns

A WKT column is recognised by its **values**, under common names (`wkt`,
`geometry`, `geom`, `shape`, ...). WKT declares its own kind per value, so one
column may mix points, lines, and polygons — which is `add_collection` territory.

In [ ]:
wkt_df = pd.DataFrame({
    "geometry": [
        "POINT (-5.36 36.13)",
        "LINESTRING (-5.44 36.05, -5.36 36.09, -5.30 36.14)",
        "POLYGON ((-5.42 36.00, -5.32 36.00, -5.32 36.06, "
        "-5.42 36.06, -5.42 36.00))",
    ],
    "label": ["Buoy", "Route", "Zone"],
})
m = Map()
m.add_collection(wkt_df, name="label", layer_group="WKT demo")
m

A `MULTILINESTRING` is one feature with disjoint parts — one sidebar entry, and no
segment joining the parts (the same holds for a `MultiLineString` from GeoPandas or
GeoJSON):

In [ ]:
legs = pd.DataFrame({
    "geometry": ["MULTILINESTRING ((-5.62 36.12, -5.52 36.12), "
                 "(-5.45 36.15, -5.35 36.15))"],
    "route": ["Ferry legs"],
})
m = Map()
m.add_line(legs, name="route", weight=4)
m

A WKT column whose name the guess would miss: point `shape_id_col` (or
`line_id_col` for lines) at it. The values being WKT is what decides — no real id
column holds `POLYGON ((...`.

In [ ]:
zones2 = pd.DataFrame({
    "boundary": ["POLYGON ((-5.62 36.02, -5.52 36.02, -5.52 36.10, "
                 "-5.62 36.10, -5.62 36.02))"],
    "risk": ["high"],
})
m = Map()
m.add_polygon(zones2, shape_id_col="boundary", name="Risk zone", color="crimson")
m

## 6. GeoPandas

Shapely geometries are x/y = **lon/lat**; swiftmap transposes for you. A geometry
column that mixes kinds goes through `add_collection`.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon

gdf = gpd.GeoDataFrame(
    {"kind": ["port", "route", "zone"]},
    geometry=[
        Point(-5.44, 36.13),
        LineString([(-5.44, 36.13), (-5.36, 36.05), (-5.25, 36.10)]),
        Polygon([(-5.60, 35.95), (-5.50, 35.95), (-5.50, 36.02), (-5.60, 36.02)]),
    ],
)
m = Map()
m.add_collection(gdf, name="kind", layer_group="GeoPandas")
m

Interior holes and MultiPolygons survive the whole way — a polygon with a hole
renders as a donut, and a MultiPolygon stays one feature and one sidebar entry.

In [ ]:
outer = [(-5.50, 36.15), (-5.30, 36.15), (-5.30, 36.30), (-5.50, 36.30)]
hole = [(-5.44, 36.19), (-5.36, 36.19), (-5.36, 36.26), (-5.44, 36.26)]
donut = gpd.GeoDataFrame({"name": ["Exclusion"]},
                         geometry=[Polygon(outer, [hole])])
m = Map()
m.add_polygon(donut, name="Exclusion", fill_color="#e15759", fill_opacity=0.5)
m

## 7. geostructures

`Coordinate` is (longitude, latitude). Any shape or collection routes by its own
type; a `GeoCircle` arrives as the polygon it describes, and a `GeoRing` as a polygon
with a real hole. A timestamped `Track`
parses the same way — animating one is the time notebook's topic.

In [ ]:
from geostructures import (Coordinate, GeoPoint, GeoLineString, GeoCircle,
                           GeoRing)
from geostructures.collections import FeatureCollection

fc = FeatureCollection([
    GeoPoint(Coordinate(-5.36, 36.13)),
    GeoLineString([Coordinate(-5.44, 36.05), Coordinate(-5.36, 36.09)]),
    GeoCircle(Coordinate(-5.30, 36.02), radius=1500),
    GeoRing(Coordinate(-5.22, 36.10), inner_radius=600, outer_radius=1200),
])
m = Map()
m.add_collection(fc, name="Survey", layer_group="geostructures")
m

## 8. GeoJSON

A dict or a JSON string; feature properties become popup fields, and `name` /
`layer_group` can reference property keys the same as DataFrame columns.

In [ ]:
geojson = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "properties": {"site": "Buoy 7"},
         "geometry": {"type": "Point", "coordinates": [-5.33, 36.07]}},
        {"type": "Feature", "properties": {"site": "Ferry route"},
         "geometry": {"type": "LineString",
                      "coordinates": [[-5.44, 36.13], [-5.31, 35.90]]}},
    ],
}
m = Map()
m.add_geojson(geojson, name="site")
m

## 9. A bare geometry, with no table around it

A WKT string, a shapely geometry, or a geostructures shape goes straight into a
builder — nothing to wrap it in when you have exactly one thing to draw.

In [ ]:
m = Map()
m.add_polygon("POLYGON ((-5.42 36.00, -5.32 36.00, -5.32 36.06, -5.42 36.06, "
              "-5.42 36.00))", name="From WKT", fill_opacity=0.3)
m.add_circle_markers(Point(-5.37, 36.03), name="From shapely", radius=9)
m

## 10. Raw lists and dicts

Coordinate lists carry no axis declaration, so swiftmap decides **once for the
whole dataset**: a first value beyond ±90° can only be a longitude (lon-first);
absent that evidence anywhere, lat-first. When your data could read both ways,
state it with `coord_order` instead of relying on the guess.

In [ ]:
pts = [[36.11, -5.47], [36.08, -5.41], [36.05, -5.35]]   # read lat-first
m = Map()
m.add_circle_markers(pts, name="Raw points")
m

In [ ]:
ring = [[-5.47, 36.11], [-5.35, 36.11], [-5.35, 36.18], [-5.47, 36.18]]
m = Map()
m.add_polygon(ring, coord_order="lon_lat", name="Raw ring")
m

## 11. When it cannot read what you passed

Nothing raises mid-chain — an exception partway through a chain of `add_*` calls
would discard the layers already added. You get a warning naming the problem and
what to do about it, and no layer.

In [ ]:
import warnings

m = Map()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    m.add_markers(12345)              # not a supported source
[str(w.message) for w in caught]

That's every source. **01_quickstart** covers the journey from here — styling,
grouping, popups, and export.